# Feature diagnostics

This notebook summarizes existing feature-set and feature-importance artifacts. It does not run inference or training.

In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.report_diagrams_utils import report_output_paths, save_report_figure, save_report_table

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
OUTPUTS = report_output_paths(DATA_ROOT)
ANALYSIS_ROOT = DATA_ROOT / "experiment-results/classification/analysis/feature-set-similarity-effect"
COMPLETE_ROOT = DATA_ROOT / "experiment-results/classification/analysis/complete-analysis-feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100/output"
IMPORTANCE_ROOT = DATA_ROOT / "experiment-results/classification/feature-importance"
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
effect = pd.read_csv(ANALYSIS_ROOT / "feature_set_effect_summary.csv")
coverage = pd.read_csv(ANALYSIS_ROOT / "feature_set_coverage.csv")
feature_window = pd.read_csv(COMPLETE_ROOT / "feature_and_window_summary.csv")
save_report_table(effect, OUTPUTS.tables / "feature_set_effect_summary.csv")
save_report_table(coverage, OUTPUTS.tables / "feature_set_coverage.csv")
save_report_table(feature_window, OUTPUTS.tables / "feature_and_window_summary.csv")

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
plot_data = effect.dropna(subset=["mean_delta"]).sort_values("mean_delta")
sns.barplot(data=plot_data, y="scorer_llm", x="mean_delta", hue="feature_set", ax=axis)
axis.set(title="Feature-set effect on macro F1", xlabel="Mean delta", ylabel="Scoring model")
save_report_figure(figure, OUTPUTS.figures / "feature_set_effect")
plt.close(figure)

In [ ]:
numeric = feature_window.select_dtypes(include="number").corr(numeric_only=True)
figure, axis = plt.subplots(figsize=(12, 10), constrained_layout=True)
sns.heatmap(numeric, cmap="vlag", center=0, ax=axis)
axis.set_title("Feature and window metric correlations")
save_report_figure(figure, OUTPUTS.figures / "feature_metric_correlation")
plt.close(figure)

In [ ]:
importance_files = sorted(IMPORTANCE_ROOT.glob("**/*.csv"))
importance_frames = []
for path in importance_files:
    if path.stat().st_size == 0:
        continue
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        continue
    if frame.empty:
        continue
    frame["source_file"] = path.parent.name
    importance_frames.append(frame)
if importance_frames:
    importance = pd.concat(importance_frames, ignore_index=True)
    save_report_table(importance, OUTPUTS.tables / "feature_importance_all.csv")
    value_column = next((column for column in ("importance_mean", "mean_importance", "importance") if column in importance), None)
    if value_column is not None:
        top = importance.nlargest(20, value_column)
        figure, axis = plt.subplots(figsize=(9, 7), constrained_layout=True)
        sns.barplot(data=top, y=top.columns[0], x=value_column, hue="source_file", ax=axis, legend=False)
        axis.set_title("Top reported feature importances")
        save_report_figure(figure, OUTPUTS.figures / "feature_importance_top20")
        plt.close(figure)